In [1]:
import csv
import time
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone

import pandas as pd

ATOM_NS = "{http://www.w3.org/2005/Atom}"
USER_AGENT = "rss-backfill-notebook/1.0 (personal research tool)"

In [8]:
#define parameters for the data collection

SUBREDDIT = "nascar"          # without r/
START_DATE = "2025-02-02"  # YYYY-MM-DD, UTC
END_DATE = "2026-06-30"    # YYYY-MM-DD, UTC
KEYWORD = "FedEx"              # sponsor here
CHUNK_DAYS = 7               # size of each date window; shrink for high-volume subs
DELAY_SECONDS = 8.0          # politeness delay between requests (raise if you still see 429s)
MAX_RETRIES = 5               # retries per chunk on a 429 before giving up on it
BACKOFF_BASE = 15.0           # seconds; used if Reddit doesn't send a Retry-After header
OUTPUT_CSV = "data/raw/reddit_posts.csv"

RESUME_AFTER = None  # YYYY-MM-DD, UTC; set to None to start from the beginning

In [3]:
def build_listing_url(subreddit: str, after: str | None = None, limit: int = 100) -> str:
    params = {"limit": limit}
    if after:
        params["after"] = after
    base = f"https://www.reddit.com/r/{urllib.parse.quote(subreddit)}/new/.rss"
    return f"{base}?{urllib.parse.urlencode(params)}"


def fetch_feed(url: str, timeout: int = 15) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def fetch_feed_with_retry(url: str, max_retries: int, backoff_base: float, timeout: int = 15) -> bytes:
    """Fetch a feed, retrying on HTTP 429 with backoff.

    Honors the Retry-After header if Reddit sends one; otherwise backs off
    exponentially starting from backoff_base seconds.
    """
    attempt = 0
    while True:
        try:
            return fetch_feed(url, timeout=timeout)
        except urllib.error.HTTPError as e:
            if e.code != 429 or attempt >= max_retries:
                raise
            retry_after = e.headers.get("Retry-After") if e.headers else None
            if retry_after is not None:
                try:
                    wait = float(retry_after)
                except ValueError:
                    wait = backoff_base * (2 ** attempt)
            else:
                wait = backoff_base * (2 ** attempt)
            attempt += 1
            print(f"    429 rate limited, waiting {wait:.0f}s before retry {attempt}/{max_retries} ... ", end="")
            time.sleep(wait)


def parse_entries(xml_bytes: bytes):
    """Parse an Atom feed\'s <entry> elements into dicts.

    The <id> element on Reddit listing feeds is the post\'s fullname
    (e.g. "t3_1abcde"), which doubles as the pagination cursor.
    """
    root = ET.fromstring(xml_bytes)
    entries = []
    for entry in root.findall(f"{ATOM_NS}entry"):
        title_el = entry.find(f"{ATOM_NS}title")
        link_el = entry.find(f"{ATOM_NS}link")
        published_el = entry.find(f"{ATOM_NS}published")
        updated_el = entry.find(f"{ATOM_NS}updated")
        author_el = entry.find(f"{ATOM_NS}author/{ATOM_NS}name")
        content_el = entry.find(f"{ATOM_NS}content")
        id_el = entry.find(f"{ATOM_NS}id")

        published = published_el.text if published_el is not None else (
            updated_el.text if updated_el is not None else None
        )

        entries.append({
            "id": id_el.text if id_el is not None else None,
            "title": title_el.text if title_el is not None else "",
            "link": link_el.get("href") if link_el is not None else "",
            "author": author_el.text if author_el is not None else "",
            "published": published,
            "content": content_el.text if content_el is not None else "",
        })
    return entries

In [4]:
start = datetime.strptime(START_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
end = datetime.strptime(END_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
assert end > start, "END_DATE must be after START_DATE"

seen_ids = set()
all_entries = []
after = RESUME_AFTER
page = 0
stopped_early = False
last_seen_date = None

print(f"Backfilling r/{SUBREDDIT} from {START_DATE} to {END_DATE} via pagination...")

while True:
    page += 1
    url = build_listing_url(SUBREDDIT, after=after)
    print(f"[page {page}] after={after} ... ", end="")

    try:
        xml_bytes = fetch_feed_with_retry(url, MAX_RETRIES, BACKOFF_BASE)
        entries = parse_entries(xml_bytes)
    except Exception as e:
        print(f"FAILED ({e}) — stopping early")
        stopped_early = True
        break

    if not entries:
        print("no entries returned — reached the end of what Reddit will paginate")
        break

    in_range = 0
    hit_start_boundary = False
    for e in entries:
        dt = pd.to_datetime(e["published"], errors="coerce")
        if pd.isna(dt):
            continue
        last_seen_date = dt
        if dt > end:
            continue  # still newer than our window, keep paging
        if dt < start:
            hit_start_boundary = True
            continue
        if e["id"] and e["id"] not in seen_ids:
            seen_ids.add(e["id"])
            all_entries.append(e)
            in_range += 1

    print(f"{len(entries)} entries, {in_range} in range (oldest on page: {last_seen_date})")

    after = entries[-1]["id"]  # fullname cursor for next page

    if hit_start_boundary:
        print("reached posts older than START_DATE — done")
        break

    time.sleep(DELAY_SECONDS)

print(f"\nDone. {len(all_entries)} unique posts collected.")
if stopped_early:
    print(f"Stopped early due to repeated failures. To resume later, set:")
    print(f"    RESUME_AFTER = \"{after}\"")
    print(f"and rerun from the parameters cell down.")

Backfilling r/nascar from 2025-02-02 to 2026-06-30 via pagination...
[page 1] after=None ... 100 entries, 0 in range (oldest on page: 2026-06-30 00:38:49+00:00)
[page 2] after=t3_1uja6lh ...     429 rate limited, waiting 15s before retry 1/5 ... 100 entries, 98 in range (oldest on page: 2026-06-27 05:16:18+00:00)
[page 3] after=t3_1ugttb1 ...     429 rate limited, waiting 15s before retry 1/5 ...     429 rate limited, waiting 30s before retry 2/5 ...     429 rate limited, waiting 60s before retry 3/5 ... 100 entries, 100 in range (oldest on page: 2026-06-24 17:15:52+00:00)
[page 4] after=t3_1uejnpl ... 100 entries, 100 in range (oldest on page: 2026-06-22 03:59:34+00:00)
[page 5] after=t3_1uca50v ...     429 rate limited, waiting 15s before retry 1/5 ...     429 rate limited, waiting 30s before retry 2/5 ... 100 entries, 100 in range (oldest on page: 2026-06-20 09:19:18+00:00)
[page 6] after=t3_1uas27j ...     429 rate limited, waiting 15s before retry 1/5 ...     429 rate limited, wai

In [5]:
df = pd.DataFrame(all_entries)
df["published"] = pd.to_datetime(df["published"], errors="coerce")
df = df.sort_values("published").reset_index(drop=True)
df.head()

,id,title,link,author,published,content
0,t3_1tzh93z,Need help finding a picture of an obscure car,https://www.reddit.com/r/NASCAR/comments/1tzh9...,/u/zplant0612,2026-06-07 16:49:17+00:00,"<!-- SC_OFF --><div class=""md""><p>Hello there,..."
1,t3_1tzj0io,Is there a subreddit for NASCAR Modified? I fe...,https://www.reddit.com/r/NASCAR/comments/1tzj0...,/u/Lukecv1,2026-06-07 17:55:46+00:00,"<!-- SC_OFF --><div class=""md""><p>Basically th..."
2,t3_1tzj4vp,Race Thread: NCS FireKeepers Casino 400 at Mic...,https://www.reddit.com/r/NASCAR/comments/1tzj4...,/u/NASCARThreadBot,2026-06-07 18:00:12+00:00,"<!-- SC_OFF --><div class=""md""><p><a href=""#th..."
3,t3_1tzk4ca,Michigan is sold out,https://www.reddit.com/r/NASCAR/comments/1tzk4...,/u/SirRelkinstein,2026-06-07 18:35:19+00:00,"<!-- SC_OFF --><div class=""md""><p>Believe this..."
4,t3_1tzk8dy,What's a paint scheme that most people seem to...,https://www.reddit.com/r/NASCAR/comments/1tzk8...,/u/Aztek-Lovett,2026-06-07 18:39:35+00:00,"<table> <tr><td> <a href=""https://www.reddit.c..."


In [6]:
def construct_df_sponsor(df: pd.DataFrame, sponsor: str, driver: str, team: str) -> pd.DataFrame:
    """Construct a DataFrame with posts containing the sponsor keyword."""
    df_filtered = df[df["title"].str.contains(sponsor, case=False, na=False) |
                     df["content"].str.contains(sponsor, case=False, na=False) |
                     df["title"].str.contains(driver, case=False, na=False) |
                     df["content"].str.contains(driver, case=False, na=False) |
                     df["title"].str.contains(team, case=False, na=False) |
                     df["content"].str.contains(team, case=False, na=False)].copy()
    return df_filtered


SPONSORS = [
    ("FedEx", "Denny Hamlin", "Joe Gibbs Racing"),
    ("Busch Light", "Ross Chastain", "Trackhouse Racing"),
    ("Cheddar\'s Scratch Kitchen", "Austin Hill", "Richard Childress Racing"),
    ("Love\'s Travel Stops", "Todd Gilliland", "Front Row Motorsports"),
    ("Castrol", "Brad Keselowski", "RFK Racing"),
]

df_sponsor_dict = {}
for sponsor, driver, team in SPONSORS:
    df_sponsor_dict[sponsor] = construct_df_sponsor(df, sponsor, driver, team)
    print(f"Posts mentioning \'{sponsor}\', \'{driver}\', or \'{team}\': {len(df_sponsor_dict[sponsor])}")

Posts mentioning 'FedEx', 'Denny Hamlin', or 'Joe Gibbs Racing': 35
Posts mentioning 'Busch Light', 'Ross Chastain', or 'Trackhouse Racing': 9
Posts mentioning 'Cheddar's Scratch Kitchen', 'Austin Hill', or 'Richard Childress Racing': 18
Posts mentioning 'Love's Travel Stops', 'Todd Gilliland', or 'Front Row Motorsports': 6
Posts mentioning 'Castrol', 'Brad Keselowski', or 'RFK Racing': 9


In [7]:
for sponsor, sdf in df_sponsor_dict.items():
    sdf["published"] = pd.to_datetime(sdf["published"], errors="coerce")
    sdf["date"] = sdf["published"].dt.date
    sdf.sort_values("published", inplace=True)
    sdf.reset_index(drop=True, inplace=True)

df_sponsor_dict["FedEx"].head()

,id,title,link,author,published,content,date
0,t3_1tzrelo,"After winning Michigan,Denny Hamlin honors kyl...",https://www.reddit.com/r/NASCAR/comments/1tzre...,/u/Helpful-Bowler6681,2026-06-07 23:27:31+00:00,"<table> <tr><td> <a href=""https://www.reddit.c...",2026-06-07
1,t3_1tzs1r0,Denny Hamlin confirms on the Prime post race s...,https://www.reddit.com/r/NASCAR/comments/1tzs1...,/u/LBHMS,2026-06-07 23:56:55+00:00,"&#32; submitted by &#32; <a href=""https://www....",2026-06-07
2,t3_1tzszem,[Gluck] Denny Hamlin says he put plans in moti...,https://www.reddit.com/r/NASCAR/comments/1tzsz...,/u/catsgr8rthanspoonies,2026-06-08 00:39:04+00:00,"<!-- SC_OFF --><div class=""md""><blockquote> <p...",2026-06-08
3,t3_1tzw71l,Denny Hamlin driving a 2013 Jordan Kyle Busch ...,https://www.reddit.com/r/NASCAR/comments/1tzw7...,/u/ApocApollo,2026-06-08 03:13:38+00:00,"<table> <tr><td> <a href=""https://www.reddit.c...",2026-06-08
4,t3_1u04l6s,"The Day After the Races - June 8, 2026",https://www.reddit.com/r/NASCAR/comments/1u04l...,/u/NASCARThreadBot,2026-06-08 11:00:07+00:00,"<!-- SC_OFF --><div class=""md""><p>Welcome to t...",2026-06-08


In [9]:
combined = []
for sponsor, sdf in df_sponsor_dict.items():
    tmp = sdf.copy()
    tmp["sponsor"] = sponsor
    combined.append(tmp)

df_sponsors_combined = pd.concat(combined, ignore_index=True)
df_sponsors_combined = df_sponsors_combined.sort_values(["sponsor", "published"]).reset_index(drop=True)
df_sponsors_combined.head()

,id,title,link,author,published,content,date,sponsor
0,t3_1tzvru0,Ross Chastain currently only has TWO top 10 fi...,https://www.reddit.com/r/NASCAR/comments/1tzvr...,/u/Gragson18GOAT,2026-06-08 02:53:08+00:00,"<table> <tr><td> <a href=""https://www.reddit.c...",2026-06-08,Busch Light
1,t3_1u0q5dj,2026 LASTCAR Cup & Truck Chase standings (Afte...,https://www.reddit.com/r/NASCAR/comments/1u0q5...,/u/TIFUthebestSubreddit,2026-06-09 00:45:36+00:00,"<!-- SC_OFF --><div class=""md""><p>Cup Series C...",2026-06-09,Busch Light
2,t3_1u1e1lk,Ross Chastain’s Busch Light Lime scheme for Po...,https://www.reddit.com/r/NASCAR/comments/1u1e1...,/u/BuschWhackerReviews,2026-06-09 18:48:48+00:00,"&#32; submitted by &#32; <a href=""https://www....",2026-06-09,Busch Light
3,t3_1u6wjun,2026 LASTCAR Cup & Xfinity Chase standings (Af...,https://www.reddit.com/r/NASCAR/comments/1u6wj...,/u/TIFUthebestSubreddit,2026-06-15 23:10:14+00:00,"<!-- SC_OFF --><div class=""md""><p>Cup Series C...",2026-06-15,Busch Light
4,t3_1u6yy3f,Better Look At Ross Chastain’s Kubota “Veteran...,https://www.reddit.com/r/NASCAR/comments/1u6yy...,/u/InsideGuard2106,2026-06-16 00:54:54+00:00,"<table> <tr><td> <a href=""https://www.reddit.c...",2026-06-16,Busch Light


In [11]:
df_sponsors_combined.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_sponsors_combined)} rows to {OUTPUT_CSV}")

Saved 77 rows to data/raw/reddit_posts.csv
